In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

from datasets import load_dataset

# ================ Load Dataset ===================
dataset = load_dataset("emotion")

# Split dataset
train_texts = list(dataset["train"]["text"])
train_labels = list(dataset["train"]["label"])
test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

In [10]:
# ================ Tokenization ===================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors="pt")


train_encodings = tokenize(train_texts)
test_encodings = tokenize(test_texts)


# ================ Custom Dataset ===================
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}, self.labels[idx]


train_dataset = EmotionDataset(train_encodings, train_labels)
test_dataset = EmotionDataset(test_encodings, test_labels)

# ================ Dataloader ===================
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)


# ================ Model Definition ===================
class EmotionClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super(EmotionClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.conv1d = nn.Conv1d(
            in_channels=embedding_dim, out_channels=64, kernel_size=3, padding=1
        )
        self.relu = nn.ReLU()
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        x = x.permute(0, 2, 1)  # (batch_size, embedding_dim, seq_len)
        x = self.conv1d(x)  # (batch_size, out_channels, seq_len)
        x = self.relu(x)
        x = self.global_avg_pool(x)  # (batch_size, out_channels, 1)
        x = x.squeeze(2)  # (batch_size, out_channels)
        x = self.fc(x)  # (batch_size, num_classes)
        return x


# Model instance
vocab_size = tokenizer.vocab_size
embedding_dim = 128  # You can choose the embedding dimension
num_classes = len(set(train_labels))
model = EmotionClassifier(vocab_size, embedding_dim, num_classes)

# ================ Training ===================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch, labels in train_loader:
        batch = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch["input_ids"])
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}"
    )

# ================ Evaluation ===================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch, labels in test_loader:
        batch = {key: val.to(device) for key, val in batch.items()}
        labels = labels.to(device)

        outputs = model(batch["input_ids"])
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")


# ================ Prediction ===================
def predict(sentence):
    model.eval()
    encoded = tokenize([sentence])
    encoded = {key: val.to(device) for key, val in encoded.items()}

    with torch.no_grad():
        output = model(encoded["input_ids"])
        prediction = torch.argmax(output, dim=1).cpu().item()

    emotions = ["sadness", "joy", "love", "anger", "fear", "surprise"]
    return emotions[prediction]


sample_sentence = "I’m so excited about this new adventure!"
print(f"Predicted Emotion: {predict(sample_sentence)}")

Epoch [1/5], Loss: 1.4675
Epoch [2/5], Loss: 0.8621
Epoch [3/5], Loss: 0.4800
Epoch [4/5], Loss: 0.2872
Epoch [5/5], Loss: 0.1842
Test Accuracy: 86.50%
Predicted Emotion: joy
